# Group the 111 Topics by Hand

| Item | Description |
|---|---|
| Goal | Record the hand-made grouping of the 111 codebook topics into 26 topic groups, check that it is a partition, and save it. The analysis in T2.4 and the validation in T2.3.2 use the groups as the upper level of the codebook. They replace the 15 broad clusters of the draft, which the review found too mixed to read. |
| Input | `codebook_v1.parquet` from T2.1.4 in the T2.1 Processed folder. The grouping itself is recorded in Step 3 of this notebook. |
| Output | `topic_groups.parquet` in the T2.1 Processed folder (`topic_id`, `label`, `group`); `table_topic_groups.csv`, `.tex`, and `.md` under `R - Results/tables/...`, the readable form of the grouping. |
| Variables | • `group`: the group name<br>• every topic appears in exactly one group |
| Run | The grouping was made by hand on 2026-10-21, from the topic labels and definitions, after the codebook was frozen. Run 2026-10-31. No model call. |


## Step 1. Import packages


In [1]:
from pathlib import Path

import pandas as pd


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries")
NOTEBOOK_NAME = "T2.1.5 - Groups - Group the 111 Topics by Hand"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"

FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Record the manual grouping

The 26 groups and the topic numbers they hold, as decided by hand from the topic labels and definitions. A topic number appears once.


In [3]:
GROUPS = {
    "Advanced computing, AI, and nanotechnology": [94, 95, 109],
    "Aviation policy, safety, and air traffic management": [137, 143, 144, 145, 146, 147],
    "Broadband infrastructure, spectrum, and universal service": [1, 47, 48, 49, 50, 91],
    "Climate and environmental regulation": [35, 36, 78],
    "Communications regulation and media": [52, 54, 55, 56, 84],
    "Competition and intellectual property": [69, 90],
    "Competitiveness, innovation, and STEM workforce": [102, 103, 115, 116, 117, 119, 121],
    "Consumer and financial protection": [53, 87, 89, 139],
    "Cybersecurity and system integrity": [61, 73, 100],
    "Energy policy, research, and clean technologies": [21, 22, 23, 24, 25, 26, 30],
    "Federal budgets, appropriations, and stimulus": [17, 18, 19, 58, 83],
    "Fossil fuels, pipelines, and fuel standards": [27, 32, 34],
    "Government operations and oversight": [2, 74, 120],
    "Hazard preparedness and emergency response": [40, 41, 138],
    "Health IT and biomedical policy": [107, 108],
    "Homeland and transportation security": [43, 62, 86, 122, 141, 150],
    "Motor carrier and vehicle safety": [79, 136, 148, 149],
    "NASA and space policy": [3, 4, 7, 9, 10, 11, 12, 13, 14, 20, 112, 125, 126],
    "Oceans, fisheries, and Arctic": [44, 45, 46],
    "Passenger rail and rail safety": [127, 128, 129],
    "Privacy and digital consumer protection": [65, 67, 68, 92, 93],
    "Science agencies, advisory bodies, and integrity": [110, 111, 114],
    "Sports and tourism regulation": [76, 134],
    "Surface transportation and infrastructure funding": [15, 130, 132, 133],
    "Trade, supply chains, and critical minerals": [29, 59, 60],
    "Weather, satellites, and Earth observation": [6, 37, 38],
}


## Step 4. Check the grouping against the codebook

Every codebook topic has exactly one group and no group names a topic outside the codebook.


In [4]:
codebook = pd.read_parquet(PROCESSED_FOLDER / "codebook_v1.parquet")
groups = pd.DataFrame([(g, t) for g, ids in GROUPS.items() for t in ids], columns=["group", "topic_id"])
assert groups["topic_id"].is_unique, "a topic appears in two groups"
assert set(groups["topic_id"]) == set(codebook["topic_id"]), "the grouping and the codebook list different topics"
merged = codebook[["topic_id", "label", "definition"]].merge(groups, on="topic_id")
print(f"{len(codebook)} topics in the codebook; every one is in exactly one of the {groups['group'].nunique()} groups")


111 topics in the codebook; every one is in exactly one of the 26 groups


## Step 5. Save the grouping


In [5]:
out = merged[["topic_id", "label", "group"]].sort_values(["group", "topic_id"]).reset_index(drop=True)
out.to_parquet(PROCESSED_FOLDER / "topic_groups.parquet", index=False)

TABLE_FOLDER.mkdir(parents=True, exist_ok=True)
out.to_csv(TABLE_FOLDER / "table_topic_groups.csv", index=False)
out.style.hide(axis="index").to_latex(TABLE_FOLDER / "table_topic_groups.tex", hrules=True, column_format="rll", environment="longtable")
lines = ["# The 26 topic groups and their topics", ""]
for group, g in out.groupby("group", sort=True):
    lines += [f"## {group} ({len(g)} topics)", ""] + [f"- {r.topic_id} {r.label}" for r in g.itertuples()] + [""]
(TABLE_FOLDER / "table_topic_groups.md").write_text("\n".join(lines), encoding="utf-8")
print(f"wrote topic_groups.parquet and table_topic_groups.csv/.tex/.md ({len(out)} rows)")


wrote topic_groups.parquet and table_topic_groups.csv/.tex/.md (111 rows)


## Step 6. Print the groups


In [6]:
summary = out.groupby("group").agg(topics=("topic_id", "size"), labels=("label", lambda s: "; ".join(s))).sort_values("topics", ascending=False)
pd.set_option("display.max_colwidth", 200)
display(summary)


,topics,labels
group,,
NASA and space policy,13,NASA Authorization Acts; International Space Station; James Webb Space Telescope; Commercial Crew Program; Mars and lunar exploration missions; Space Launch System; Commercial Space Launch Legisla...
"Competitiveness, innovation, and STEM workforce",7,CHIPS and Science Act; America COMPETES Act; Manufacturing Extension Partnership and related initiatives; science competitiveness legislation; Advanced Technology Programs; Small Business Innovati...
"Energy policy, research, and clean technologies",7,Nuclear Energy Research and Development; Clean Power Plan; Department of Energy research and development programs; Energy Policy Act of 2005; Smart Grid and Grid Modernization; Renewable Energy Te...
"Aviation policy, safety, and air traffic management",6,NTSB and aviation safety investigations; Aviation safety and certification policy; Commercial airline pilot fatigue and training standards; FAA reauthorization legislation; Essential Air Service; ...
"Broadband infrastructure, spectrum, and universal service",6,Universal Service Fund; Rural broadband and telecommunications infrastructure; Broadband deployment and infrastructure; spectrum policy; Public Safety Broadband Network; Digital equity and access ...
Homeland and transportation security,6,Maritime Security and Port Safety; Department of Homeland Security legislation and operations; passenger fees and charges; Transportation Worker Identification Credential program; Aviation and Tra...
Communications regulation and media,5,Telecommunications Act of 1996; FCC telecommunications regulation authority; Digital Television transition; media ownership concentration; Internet access taxation and net neutrality
"Federal budgets, appropriations, and stimulus",5,Science and research funding; Federal agency budget requests and proposals; Federal science and technology agency budgets; American Recovery and Reinvestment Act; R&D tax credits
Privacy and digital consumer protection,5,Consumer privacy and anti-spam legislation; Digital identity verification and protection; consumer data privacy legislation; Children's online privacy and safety legislation; Online advertising an...
